<div style="text-align: center;">
    <img src="ressource/ki_ag_main.jpg" alt="Logo" style="width: 1000px;"/>
</div>

# AI Agents mit n8n erstellen

n8n ist eine Open-Source Workflow-Automatisierungsplattform, mit dem wir KI Modelle zu "Agenten" machen. Das bedeutet automatisierte Workflows erstellen, die die KI automatisiert ausführen kann und ihr die notwendigen Tools und Ressourcen geben, damit sie "autonom" handeln kann wenn ein Event getriggered wird wie z.B. eine Texteingabe an das Sprachmodell, mit der Aufforderung eine Aufgabe zu erledigen oder auch Daten für KI-Modelle sammeln zu lassen, Ergebnisse weiterzuleiten oder Aktionen basierend auf KI-Entscheidungen auszulösen.

## Setup n8n

Um n8n local auf unserem server zu starten, verwenden wir den Befehl `npx n8n`, dann akzeptieren wir den Download der notwendigen packages mit y und starten, sobald bereit, den Server mit Eingabe von o im Terminal. Meldet euch an und klickt dann auf create workflow from scratch. Nun befindet ihr euch im Workflow Editor.

## Erstellen der Google Credentials

Da wir die Google Dienste für unser Projekt verwenden, brauchen wir zuerst die notwendigen Credentials für die APIs, die wir später für unseren Workflow verwenden. Navigiere dazu zur Google Cloud Console und registriere dich mit einem neuen Benutzer: https://console.cloud.google.com/apis/library?hl=de&invt=Abur8w (achte darauf auf Anmelden zu klicken und nicht auf kostenlos testen)

**1. Neues Projekt anlegen:** 
- oben rechts neben dem Google Cloud Logo auf Projekt auswählen klicken
- im erscheinenden Modal auf Neues Projekt klicken oben rechts
- dann umbennenen zu n8n und auf erstellen klicken
- nun das Projekt auswählen (es sollte danach neben dem Google Cloud Logo angezeigt werden mit dem namen n8n)

**2. Die notwendigen APIs hinzufügen:**
- Über die Suchleiste folgende APIs suchen und aktivieren: `Google Drive API`, `Google Calendar API`, `Gmail API`, `Google Sheets API`, `Google Docs API`
- Die Dienste sind aktiviert sobald man auf den aktivieren Button klickt mehr ist nicht notwendig
- Nachdem alle notwendigen APIs aktiviert wurden sollten sie links im APIs und Dienste Menü unter dem Reiter Aktivierte APIs und Dienste aufgelistet werden

**3. OAuth Anmeldedaten:**
- links den Menüpunkt OAuth-Zustimmungsbildschirm auswählen
- auf den Button erste Schritte klicken
- bei App-Informationen den Anwendungsnamen n8n eingeben und als Nutzersupport-E-Mail die eigene bereits aufgelistete E-Mail auswählen
- Zielgruppe Extern auswählen
- Kontaktdaten die bereits existierende Mail auswählen
- Nutzungsdaten akzeptieren und auf erstellen klicken

**4. Zielgruppe festlegen:**
- Links zum Menüpunkt Zielgruppe navigieren
- Auf Anwendung veröffentlichen Button klicken und bestätigen

**5. Client erstellen:**
- Links zum Menüpunkt Clients navigieren
- Im oberen Reiter auf Client erstellen klicken
- Als Anwendungstyp Webanwendung auswählen
- Als autorisierte Weiterleitungs-Uris diese hinzufügen: http://localhost:5678/rest/oauth2-credential/callback
- Auf erstellen klicken
- Nun stehen euch die `Client-ID` und der `Clientschlüssel` zur Verfügung, diese könnt ihr euch in die Zwischenablage kopieren, wir werden diese später in n8n als Credentials verwenden. Zuerst die Zwischenablage aktivieren `Windows Taste` + `V`. Sobald diese aktiviert ist, könnt ihr alles bereits kopierte erneut abrufen. Kopiert dann die Client-Id und den Client-Schlüssel in die Zwischenablage.

## Gemini API Key

Wir verwenden dieses mal Gemini 2.0 flash als Basis LLM. Dieses ist kostenlos, leistungsstark und schnell und eignet sich sehr gut um mit dem Google Ökosystem zu interagieren.
- Besuche hierzu die Google AI Studio Website: https://aistudio.google.com/apikey 
- Dort den API Key in die Zwischenablage kopieren, wir werden diesen später in den Credentials verwenden




# Kapitel 1: KI-Terminkalender-Assistent

In diesem Kapitel bauen wir einen AI Agent, der auf Anfragen (z.B. "Welche Termine stehen diese Woche an?") die nötigen Daten über die Google API holt und eine informative Zusammenfassung gibt, sowie neue Events für euch anlegen kann.

## 1.1 Trigger des Workflows

Wir benötigen etwas, das unseren Workflow triggered. Dazu klicken wir im ersten Schritt auf das Plus in unserem Workflow. Wie in der Abbildung unten zu sehen ist, öffnet sich ein Seitenmenü, aus dem man unterschiedliche Events auswählen kann. Wenn man auf das Plus klickt öffnet sich ein Seitenmenü. Lese dir alle Unterpunkte und ihre Funktionen gründlich durch. 




<div style="text-align: center;">
    <img src="n8n_ressource/trigger.png" alt="Logo" style="width: 1000px;"/>
</div>


Wir möchten dass unser Agent durch unsere Frage an ihn getriggered wird, weshalb wir hier ```on chat message``` auswählen. Es öffnet sich nun ein Pop-Up Fenster mit den Reitern **Parameters** und **Settings**. Wie man sehen kann gibt es bei Chat Message relativ wenig Optionen, weshalb wir erstmal durch Klicken auf `Back to canvas` oder klicken außerhalb des Fensters zurück zur Übersicht unserer Nodes gelangen.
Klicke nun auf ``Open Chat`` und sende eine Test Message: "Hello World"
Wie du sehen kannst wird eine Error Message erscheinen, wir müssen eine Chat Model sub-node hinzufügen.

## 1.2 AI Agent anlegen

Sobald wir ein Trigger Event ausgewählt haben, haben wir die Möglichkeit über das Plus auszuwählen, was als nächstes passieren soll. Es gibt unterschiedliche Möglichkeiten, lese dir diese gründlich durch. Wie wählen jetzt unter dem Menü Punkt `Advanced AI` den Punkt `AI Agent` aus. Sobald wir ihn ausgewählt haben können wir das pop up fenster schließen und oben links über die Zurücktaste oder durch Klicken außerhalb des Fensters, zur Übersicht navigieren. Diese sollte jetzt folgendermaßen aussehen

<div style="text-align: center;">
    <img src="n8n_ressource/agent.png" alt="Logo" style="width: 1000px;"/>
</div>


Diese AI Agent Node kann man sich wie das Gehirn oder den zentralen Koordinator des intelligenten Assistenten oder Chatbots direkt im n8n-Workflow vorstellen. Sie kann komplexe Aufgaben oder Konversationen steuern. 

### Was macht die AI Agent Node genau?

1. Empfängt Input: Er nimmt eine Aufgabe oder eine Nachricht entgegen (im Bild kommt sie vom Trigger "When chat message received").
2. Nutzt ein Sprachmodell: Er braucht zwingend ein verbundenes Chat Model. Das ist die eigentliche "Intelligenz", die Sprache versteht, plant und Antworten generiert.
3. Verwendet optional Tools: Über den Tool-Eingang kann man ihm "Werkzeuge" geben. Das können andere n8n-Nodes sein (z.B. ein HTTP Request, um Webseiten abzufragen, ein Datenbank-Node oder eben ein MCP Client Node, um mit MCP-Servern zu sprechen). Der AI Agent entscheidet selbst, basierend auf der Aufgabe und den Beschreibungen der Tools, ob und welches Tool er wann nutzen muss, um die Aufgabe zu lösen.
4. Nutzt optionales Gedächtnis: Über den Memory-Eingang kann er sich an vorherige Teile des Gesprächs erinnern, was für längere Dialoge oder Chatbots wichtig ist.
5. Gibt ein Ergebnis aus: Am Ende liefert er eine Antwort, eine Zusammenfassung oder das Ergebnis seiner Arbeit an den nächsten Node weiter.

Die AI Agent Node ist ein mächtiger Baustein, um Sprachmodelle dazu zu bringen, nicht nur zu reden, sondern über bereitgestellte Tools aktiv zu handeln, Informationen zu sammeln und komplexe Aufgaben zu lösen, die über reines Textverständnis hinausgehen. Das kleine Warndreieck im Bild zeigt übrigens an, dass die Node noch nicht vollständig konfiguriert ist. Deshalb fügen wir als erstes ein KI Modell hinzu.

### Chat Modell auswählen

Klicke auf das Plus bei `Chat Model` dort kannst du unterschiedliche Chat Modelle auswählen. In unserem Beispiel verwenden wir das Google Gemini Chat Model. 
Füge unter `Credential to connect with` unter `Create new credential` die Credentials deines Google Gemini Accounts hinzu und füge den zuvor in die Zwischenablage kopierten API Key unter `API Key` ein, speichere die Änderungen ab.
Wähle als Model ``gemini-2.0-flash-001`` aus
Teste nun die Chat Funktion in n8n, und frage nach den Terminen für diese Woche. 

Das LLM wird die Anwort nicht wissen, wir müssen ihm noch die notwendigen echten Daten zukommen lassen

### Memmory hinzufügen

Wir möchten, dass sich unser Sprachmodell an Dinge erinnern kann, dazu klicken wir in der Übersicht bei unserer AI Agent Node auf das Plus und fügen eine `Simple Memory` Funktion hinzu, damit sich das LLM an die letzten Nachrichten erinnern kann.

### Tool Create Event hinzufügen

Nun fügen wir das notwendige Tool hinzu, damit das LLM neue Events anlegen kann. 
- Klicke auf das Plus Icon bei Tool und suche nach Google Calendar
- Es öffnet sich wieder ein Pop-Up Fenster.

**Einstellungen im Pop-Up Fenster**

**Input und Output:**
- Links können wir unseren Input sehen, wir sehen die Nachricht die wir bereits gesendet haben in Key Value Paaren
- Rechts sehen wir den Output, dieser ist Zurzeit noch leer, da wir noch keine Anfragen an die Google Calender API getätigt haben

**Parameters:**

In der Mitte sehen wir wieder die Parameter. Ändere den Namen der Node von Google Calendar zu Create Event
1. Credentials:
- Zuerst müssen wir wieder unsere Google Credentials anlegen, diese erstellen wir wieder mit ``Create New Credentials``. Kopiere die Client-ID (welche mit .com endet) aus deiner Zwischenablage in das Feld ``Client-ID`` und kopiere das Client-Secret aus der Zwischenablage in das Feld ``Client Secret`` und klicke auf ``Save``. Melde dich anschließend mit deinem Google Account an. Falls erfolgreich sollte ein Grünes Pop-Up mit Account connected erscheinen.
2. Tool Description:
- Wir können eine Description für das Tool entweder automatisch erstellen, oder manuell festlegen. Diese gibt der KI Kontext zum Tool und ist auch praktisch um Regeln festzulegen, wie die KI das Tool verwenden soll. wir lassen diese zuerst einmal auf Set Automatically
3. Operation:
- Definiert was wir genau machen wollen im Google Calendar. Wir lassen dies erstmal auf Create um ein neues Event zu erstellen
4. Calendar:
- dort wählen wir unseren eigenen aus der Liste aus
5. Start:
- der Startzeitpunkt des angelegten Events, diesen lassen wir so wie er ist
6. End:
- der Endzeitpunkt des Events,  diesen lassen wir so wie er ist
7. Additional Fields:
- die weitere Parameter die wir angeben können um unserem angelgtem Kalenderevent mehr Details zu geben. Hier fügen wir erstmal keine weiteren hinzu.

Versuche nun im Chat Fenster einen neuen Termin für Samstag den 26.04.2025 um 18:30 Uhr anzulegen namens Sportschau. Überprüfe im Google Calendar ob das Event korrekt angelegt wurde.






In [ ]:
# Wurde das Event korrekt angelegt? Was fällt dir auf?

Das Event wurde nicht korrekt angelegt, das liegt daran, dass das Start und Enddatum nicht variabel sind und keine zusätzlichen Parameter definiert wurden:
- Start ist mit `$now` definiert, hier wird das aktuelle Datum und die aktuelle Uhzeit verwendet, es ist fix und wird nicht variabel durch das LLM angelegt
- Um es Variabel zu machen klocken wir bei Start auf Expression und geben dann ``{{ $fromAI("Start", "It is the meeting Starting Time") }}`` ein. `$fromAI` bedeutet die Parameter werden vom LLM übergeben. Das LLM ist schlau genug sich den Parameter `Start` aus dem Chat Input herauszufiltern, die Description "It is the meeting Starting Time" gibt nochmal zusätzlichen Kontext.
- Wende dieses Schema nun auch bei dem `End` Parameter an
- Füge in `Additional Fields` noch die Parameter `Description` für den Titel des Events hinzu und `Summary` für die Description des Events hinzu

<div style="text-align: center;">
    <img src="n8n_ressource/parameters.png" alt="Logo" style="width: 1500px;"/>
</div>


Auf dem Screenshot kann man erkennen, dass das LLM korrekt die Parameter aus dem Chat Input herausgefiltert hat und links im Input Fenster zugeordnet hat. Rechts kann man dann erkennen, dass diese Parameter an die Google API übergeben wurden.

Nun müssen wir auch noch den System Prompt des AI Agents anpassen, klicke Doppelklick auf den AI Agent und füge unter **Add Option** eine **System Message hinzu**. Ändere die System Message von Fixed auf Expression. Damit können Variablen übergeben werden wie zum Beispiel das aktuelle Datum mit `{{$now}}` im Eingabefeld gibt es einen Erweiterungsbutton des Fenster klicke darauf und kopiere dann folgende Expression rein:

In [ ]:
You are a helpful assistant and respond in the language that the user speaks. The current date is {{ $now }}. The timezone is Europe/Berlin. 

# Rules

1. Use the LLM when no calendar-related query is present. When asked to create an event, use the Create Event tool — unless specified otherwise, the default event duration is one hour, always add an offset of minus 4 hours! 

2. Important: All time values must be created in the correct timezone (Europe/Berlin). Google receives it in the American/New York timezone, therefore add an offset of -4 hours for every event.

3. When asked about an event or a specific date, use the Get Events tool.



### Tool Get Events hinzufügen

- Füge beim AI Agent unter dem Plus bei Tools eine neue Google Calendar Node hinzu
- Setze die Operaiton auf Get Many
- Altviere den Parameter **Return all**
- Lasse das LLM das Startdatum festlegen indem du After auf Expression setzt und `{{ $fromAI("after") }}` eingibst.

Nun haben wir einen simplen AI Agent der auf Befehl Termine anlegen und finden kann. 

# Kapitel 2: Täglicher Newsletter Service

Nun möchten wir einen automatisierten Newsletter Workflow erstellen, der den Subscribern unserer E-Mail Liste automatisiert jeden Morgen die neuesten Workflow Templates von n8n.io mit KI zusammenfasst und über Gmail an ihre Mails schickt. Dabei können den Subscribern Kategorien von n8n zugeordnet werden, welche sie abonniert haben (zb KI, Sales, etc.)

Der Workflow ist unvollständig. Ergänze ihn und mache ihn lauffähig.

### Setup

- erstelle einen neuen Workflow in n8n, indem du auf das Plus oben links klickst neben dem n8n Logo
- im neuen Projekt oben rechts in der Header Leiste auf die 3 Punkte klicken, im dropdown Menü dann die ``n8n_newsletter.json`` Datei importieren, welche du unter n8n_ressource


